### **Laboratorio 2 - Inferencia, decoding y KV cache**

#### **Propósito y restricción temporal**

Las exposiciones se realizan **al final**, después de cerrar el trabajo experimental.

Cronograma máximo:

| Minutos | Etapa | Duración |
|---:|---|---:|
| 0-10 | START | 10 min |
| 10-30 | BUILD | 20 min |
| 30-60 | EVALUATE A | 30 min |
| 60-80 | EVALUATE B | 20 min |
| 80-95 | VALIDATE/OPTIONAL | 15 min |
| 95-100 | CRITIQUE/cierre del laboratorio | 5 min |
| 100-110 | BREAK | 10 min |
| 110-210 | EXPOSE/DEFEND | hasta 100 min |

La sesión puede terminar antes.

#### **Preparación previa**

La búsqueda bibliográfica y la preparación de diapositivas se realizan antes del jueves.

Herramientas posibles:

```text
Elicit/SciSpace/Consensus
ResearchRabbit/Connected Papers
Scite
```

La evidencia final debe verificarse contra fuente primaria.

Durante el laboratorio solo se presenta y defiende el trabajo ya preparado.

#### **1. START**

Registra:

- pregunta,
- hipótesis,
- baseline,
- variable modificada,
- variables constantes,
- métrica,
- criterio de interpretación.

Regla:

```text
pregunta -> hipótesis -> baseline -> una modificación -> métrica -> resultado -> limitación -> conclusión
```

#### **2. Configuración**

In [ ]:
import json
import math
import random
import statistics
import time
from typing import Optional

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo:", DEVICE)

#### **3. BUILD**

Implementa y verifica:

```text
logits
  ->
temperature
  ->
top-k/top-p
  ->
softmax
  ->
sampling
```

Valores neutrales:

```text
temperature = 1.0
top_k = desactivado
top_p = 1.0
```

In [ ]:
def entropy_bits(
    probs_1d: torch.Tensor,
) -> float:
    p = probs_1d.clamp_min(1e-12)
    return float(
        -(p * torch.log2(p)).sum().item()
    )


def apply_temperature(
    logits_1d: torch.Tensor,
    temperature: float,
) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError(
            "temperature debe ser > 0"
        )
    return logits_1d / temperature


def apply_top_k(
    logits_1d: torch.Tensor,
    top_k: Optional[int],
) -> torch.Tensor:
    if top_k is None or top_k <= 0:
        return logits_1d.clone()

    k = min(
        top_k,
        logits_1d.numel(),
    )
    values, _ = torch.topk(
        logits_1d,
        k=k,
    )
    threshold = values[-1]

    return torch.where(
        logits_1d < threshold,
        torch.full_like(
            logits_1d,
            float("-inf"),
        ),
        logits_1d,
    )


def apply_top_p(
    logits_1d: torch.Tensor,
    top_p: Optional[float],
) -> torch.Tensor:
    if top_p is None or top_p >= 1.0:
        return logits_1d.clone()
    if top_p <= 0:
        raise ValueError(
            "top_p debe estar en (0, 1]"
        )

    sorted_logits, sorted_indices = torch.sort(
        logits_1d,
        descending=True,
    )

    sorted_probs = torch.softmax(
        sorted_logits,
        dim=-1,
    )
    cumulative = torch.cumsum(
        sorted_probs,
        dim=-1,
    )

    remove = cumulative > top_p
    remove[1:] = remove[:-1].clone()
    remove[0] = False

    sorted_logits = sorted_logits.masked_fill(
        remove,
        float("-inf"),
    )

    filtered = torch.full_like(
        logits_1d,
        float("-inf"),
    )
    filtered.scatter_(
        0,
        sorted_indices,
        sorted_logits,
    )

    return filtered


def build_sampling_distribution(
    logits_1d: torch.Tensor,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
) -> torch.Tensor:
    scaled = apply_temperature(
        logits_1d,
        temperature,
    )
    filtered = apply_top_k(
        scaled,
        top_k,
    )
    filtered = apply_top_p(
        filtered,
        top_p,
    )
    return torch.softmax(
        filtered,
        dim=-1,
    )


def sample_token(
    logits_1d: torch.Tensor,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
) -> int:
    probs = build_sampling_distribution(
        logits_1d,
        temperature,
        top_k,
        top_p,
    )
    return int(
        torch.multinomial(
            probs,
            num_samples=1,
        ).item()
    )

#### **4. Tests mínimos**

In [ ]:
test_logits = torch.tensor(
    [4.0, 2.5, 1.5, 0.5, -0.5],
    dtype=torch.float32,
)

assert torch.allclose(
    apply_temperature(
        test_logits,
        1.0,
    ),
    test_logits,
)

filtered_k = apply_top_k(
    test_logits,
    2,
)
assert int(
    torch.isfinite(filtered_k).sum()
) <= 2

filtered_p = apply_top_p(
    test_logits,
    0.8,
)
assert int(
    torch.isfinite(filtered_p).sum()
) >= 1

for cfg in [
    {"temperature": 0.7},
    {"top_k": 2},
    {"top_p": 0.8},
]:
    p = build_sampling_distribution(
        test_logits,
        **cfg,
    )
    assert torch.allclose(
        p.sum(),
        torch.tensor(1.0),
        atol=1e-6,
    )

print("Tests de BUILD: OK")

#### **5. EVALUATE A**

Pregunta:

> ¿Cómo cambia la distribución y la trayectoria de generación cuando se modifica una sola política de decoding?

Se distinguen dos niveles de evidencia:

```text
distribución inicial -> mismo estado <bos> ->determinista para una estrategia trayectoria generada
  -> estados visitados dependen del sampling -> puede variar entre repeticiones
```

Se realizarán 12 repeticiones por estrategia.

No basta reportar medias. Se reportará también desviación estándar.

#### **6. Modelo causal didáctico**

In [ ]:
toy_vocab = [
    "<bos>",
    "los",
    "modelos",
    "usan",
    "contexto",
    "memoria",
    "eficiente",
    ".",
]

# Logits de transición didácticos.
# Cada fila representa el token actual y cada columna el siguiente token.
#
# Nota de diseño: la versión inicial producía distribuciones demasiado
# concentradas en varias transiciones. Con la seed usada en el ejemplo,
# temperature, top-k y top-p podían terminar generando la misma secuencia,
# aunque los mecanismos de decoding fueran distintos.
#
# Esta versión escala uniformemente los logits a la mitad. El escalado
# conserva el argmax de cada fila y, por tanto, mantiene el mismo recorrido
# bajo greedy decoding, pero reduce la concentración de la distribución.
# Esto hace más observable el efecto de distintas políticas de sampling
# en un ejemplo pequeño y reproducible.
#
# Matemáticamente, softmax(0.5 * z) es equivalente a aplicar temperatura
# T=2 sobre los logits originales. Aquí se usa solo para calibrar la
# distribución base de este modelo didáctico antes del experimento.
# No constituye una recomendación de temperatura para un modelo real.
transition_logits = 0.5 * torch.tensor([
    [-4.0,  4.0,  1.0, -2.0, -2.0, -2.0, -2.0, -3.0],
    [-4.0, -2.0,  4.0,  1.0,  0.0, -1.0, -1.0, -2.0],
    [-4.0, -2.0, -1.0,  4.0,  1.0,  0.5, -1.0, -2.0],
    [-4.0, -2.0, -2.0, -1.0,  3.0,  2.7,  0.5, -2.0],
    [-4.0, -2.0, -2.0, -2.0, -1.0,  2.5,  3.0,  1.0],
    [-4.0, -2.0, -2.0, -2.0,  1.0, -1.0,  3.2,  1.0],
    [-4.0, -2.0, -2.0, -2.0, -1.0, -1.0, -1.0,  4.0],
    [-4.0,  2.0,  1.5, -2.0, -2.0, -2.0, -2.0, -1.0],
], dtype=torch.float32)


def generate_toy_ids(
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
) -> list[int]:
    token_id = 0
    ids = [token_id]

    for _ in range(max_new_tokens):
        token_id = sample_token(
            transition_logits[token_id],
            temperature,
            top_k,
            top_p,
        )
        ids.append(token_id)

    return ids


def decode_toy(
    ids: list[int],
) -> str:
    return " ".join(
        toy_vocab[i]
        for i in ids
    )

In [ ]:
def generate_toy_trace(
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
) -> dict:
    token_id = 0
    ids = [token_id]
    entropies = []
    candidate_counts = []

    for _ in range(max_new_tokens):
        probs = build_sampling_distribution(
            transition_logits[token_id],
            temperature,
            top_k,
            top_p,
        )

        entropies.append(
            entropy_bits(probs)
        )
        candidate_counts.append(
            int((probs > 0).sum().item())
        )

        token_id = int(
            torch.multinomial(
                probs,
                num_samples=1,
            ).item()
        )
        ids.append(token_id)

    return {
        "ids": ids,
        "entropies": entropies,
        "candidate_counts": candidate_counts,
    }


def generate_toy_ids(
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
) -> list[int]:
    # Wrapper conservado para ejemplos que solo necesitan la trayectoria.
    return generate_toy_trace(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
    )["ids"]

#### **7. Métricas de trayectoria**

In [ ]:
def distinct_n(
    ids: list[int],
    n: int,
) -> float:
    if len(ids) < n:
        return 0.0

    ngrams = [
        tuple(ids[i:i+n])
        for i in range(
            len(ids) - n + 1
        )
    ]

    return len(set(ngrams))/len(ngrams)


def repeated_bigram_ratio(
    ids: list[int],
) -> float:
    if len(ids) < 2:
        return 0.0

    bigrams = [
        tuple(ids[i:i+2])
        for i in range(
            len(ids) - 1
        )
    ]

    repeated = sum(
        1
        for bg in bigrams
        if bigrams.count(bg) > 1
    )

    return repeated / len(bigrams)

#### **8. Experimento A**

Se comparan:

```text
Baseline
temperature = 1.0

Temperature low
temperature = 0.7

Temperature high
temperature = 1.3

Top-k

Top-p
```

La entropía inicial se calcula sobre `transition_logits[0]`.

La entropía media de trayectoria se calcula sobre las distribuciones efectivamente visitadas durante cada generación.

In [ ]:
configs_a = [
    {
        "nombre": "Baseline sampling",
        "temperature": 1.0,
        "top_k": None,
        "top_p": None,
    },
    {
        "nombre": "Temperature=0.7",
        "temperature": 0.7,
        "top_k": None,
        "top_p": None,
    },
    {
        "nombre": "Temperature=1.3",
        "temperature": 1.3,
        "top_k": None,
        "top_p": None,
    },
    {
        "nombre": "Top-k=3",
        "temperature": 1.0,
        "top_k": 3,
        "top_p": None,
    },
    {
        "nombre": "Top-p=0.85",
        "temperature": 1.0,
        "top_k": None,
        "top_p": 0.85,
    },
]

rows = []
initial_rows = []
outputs_by_strategy = {}

for cfg in configs_a:
    outputs_by_strategy[cfg["nombre"]] = []

    # Propiedad local del primer paso real: <bos> -> transition_logits[0].
    initial_probs = build_sampling_distribution(
        transition_logits[0],
        cfg["temperature"],
        cfg["top_k"],
        cfg["top_p"],
    )

    initial_rows.append({
        "estrategia": cfg["nombre"],
        "entropia_inicial": entropy_bits(initial_probs),
        "candidatos_iniciales": int(
            (initial_probs > 0).sum().item()
        ),
    })

    # Propiedades de trayectoria: dependen de los estados visitados.
    for rep in range(12):
        torch.manual_seed(SEED + rep)

        trace = generate_toy_trace(
            max_new_tokens=20,
            temperature=cfg["temperature"],
            top_k=cfg["top_k"],
            top_p=cfg["top_p"],
        )

        ids = trace["ids"]
        text = decode_toy(ids)
        outputs_by_strategy[cfg["nombre"]].append(text)

        rows.append({
            "estrategia": cfg["nombre"],
            "repeticion": rep,
            "entropia_media_trayectoria": float(
                np.mean(trace["entropies"])
            ),
            "candidatos_medios_trayectoria": float(
                np.mean(trace["candidate_counts"])
            ),
            "distinct_1": distinct_n(ids, 1),
            "distinct_2": distinct_n(ids, 2),
            "repeticion_bigramas": repeated_bigram_ratio(ids),
        })

df_a = pd.DataFrame(rows)
df_initial = pd.DataFrame(initial_rows)

summary_a = (
    df_a
    .groupby("estrategia")
    .agg(
        entropia_trayectoria_mean=(
            "entropia_media_trayectoria",
            "mean",
        ),
        entropia_trayectoria_std=(
            "entropia_media_trayectoria",
            "std",
        ),
        candidatos_trayectoria_mean=(
            "candidatos_medios_trayectoria",
            "mean",
        ),
        candidatos_trayectoria_std=(
            "candidatos_medios_trayectoria",
            "std",
        ),
        distinct_1_mean=("distinct_1", "mean"),
        distinct_1_std=("distinct_1", "std"),
        distinct_2_mean=("distinct_2", "mean"),
        distinct_2_std=("distinct_2", "std"),
        repeticion_mean=("repeticion_bigramas", "mean"),
        repeticion_std=("repeticion_bigramas", "std"),
        n=("repeticion", "count"),
    )
    .reset_index()
    .merge(
        df_initial,
        on="estrategia",
        how="left",
    )
)

summary_a["salidas_unicas_ratio"] = [
    len(set(outputs_by_strategy[name]))
    / len(outputs_by_strategy[name])
    for name in summary_a["estrategia"]
]

summary_a

In [ ]:
for nombre, outputs in outputs_by_strategy.items():
    print("\n", nombre)
    for text in outputs[:3]:
        print("  ", text)

#### **9. Interpretación del Experimento A**

Responde:

1. ¿Qué estrategia modifica más la entropía inicial?
2. ¿La entropía media de trayectoria coincide necesariamente con la entropía inicial?
3. ¿Qué estrategias presentan mayor dispersión entre repeticiones?
4. ¿Una diferencia de medias es grande respecto de su desviación estándar?
5. ¿Menor repetición implica necesariamente mayor calidad?
6. ¿Qué conclusión está respaldada por estas 12 repeticiones?.

#### **10. EVALUATE B**

Pregunta:

> ¿Cómo cambia el payload lógico estimado del KV cache bajo diferentes configuraciones?.

Modelo:

$$
M_{\mathrm{KV}}
=
B L T \, 2 H_{\mathrm{KV}} d_h b.
$$

No confundir:

```text
payload lógico estimado != memoria GPU total !=latencia != throughput
```

In [ ]:
def kv_cache_bytes(
    batch_size: int,
    num_layers: int,
    context_length: int,
    num_kv_heads: int,
    head_dim: int,
    bytes_per_value: int,
) -> int:
    return (
        batch_size
        * num_layers
        * context_length
        * 2
        * num_kv_heads
        * head_dim
        * bytes_per_value
    )


def swa_cache_bytes(
    batch_size: int,
    num_layers: int,
    context_length: int,
    window_size: int,
    num_kv_heads: int,
    head_dim: int,
    bytes_per_value: int,
) -> int:
    effective_context = min(
        context_length,
        window_size,
    )

    return kv_cache_bytes(
        batch_size,
        num_layers,
        effective_context,
        num_kv_heads,
        head_dim,
        bytes_per_value,
    )


def mla_cache_bytes(
    batch_size: int,
    num_layers: int,
    context_length: int,
    latent_rank: int,
    bytes_per_value: int,
) -> int:
    return (
        batch_size
        * num_layers
        * context_length
        * 2
        * latent_rank
        * bytes_per_value
    )


def to_gb(value: int) -> float:
    return value / 1_000_000_000


def to_gib(value: int) -> float:
    return value / (1024 ** 3)

In [ ]:
cfg = {
    "batch_size": 1,
    "num_layers": 32,
    "context_length": 131072,
    "query_heads": 32,
    "head_dim": 128,
    "bytes_per_value": 2,
    "gqa_kv_heads": 8,
    "swa_window": 4096,
    "mla_rank": 512,
}

values = {
    "MHA": kv_cache_bytes(
        cfg["batch_size"],
        cfg["num_layers"],
        cfg["context_length"],
        cfg["query_heads"],
        cfg["head_dim"],
        cfg["bytes_per_value"],
    ),
    "GQA": kv_cache_bytes(
        cfg["batch_size"],
        cfg["num_layers"],
        cfg["context_length"],
        cfg["gqa_kv_heads"],
        cfg["head_dim"],
        cfg["bytes_per_value"],
    ),
    "SWA": swa_cache_bytes(
        cfg["batch_size"],
        cfg["num_layers"],
        cfg["context_length"],
        cfg["swa_window"],
        cfg["query_heads"],
        cfg["head_dim"],
        cfg["bytes_per_value"],
    ),
    "MLA conceptual": mla_cache_bytes(
        cfg["batch_size"],
        cfg["num_layers"],
        cfg["context_length"],
        cfg["mla_rank"],
        cfg["bytes_per_value"],
    ),
}

df_b = pd.DataFrame([
    {
        "variante": name,
        "GB": to_gb(value),
        "GiB": to_gib(value),
        "ratio_vs_MHA": (
            value / values["MHA"]
        ),
    }
    for name, value in values.items()
])

df_b

In [ ]:
mha = values["MHA"]
gqa = values["GQA"]

assert math.isclose(
    gqa / mha,
    8 / 32,
    rel_tol=1e-12,
)

assert all(
    math.isfinite(v)
    and v > 0
    for v in values.values()
)

print("Invariantes de Experimento B: OK")

#### **11. Bytes por token y precisión**

Una segunda vista es:

$$
M_{\mathrm{KV/token}}
=
L \, 2 H_{\mathrm{KV}} d_h b
$$

para `batch=1`.

La variante de 8 bits es **idealizada**. Una implementación real puede requerir escalas, metadatos, alineamiento u otros buffers.

In [ ]:
def kv_bytes_per_token(
    num_layers: int,
    num_kv_heads: int,
    head_dim: int,
    bytes_per_value: int = 2,
) -> int:
    # K y V para un token, batch=1.
    return (
        num_layers
        * 2
        * num_kv_heads
        * head_dim
        * bytes_per_value
    )


rows_token = []
for name, kv_heads in [
    ("MHA", 32),
    ("GQA", 8),
    ("MQA", 1),
]:
    value = kv_bytes_per_token(
        num_layers=32,
        num_kv_heads=kv_heads,
        head_dim=128,
        bytes_per_value=2,
    )
    rows_token.append({
        "variante": name,
        "bytes_por_token": value,
        "KiB_por_token": value / 1024,
    })

pd.DataFrame(rows_token)

In [ ]:
precision_rows = []

for precision, bytes_per_value in [
    ("fp32", 4),
    ("fp16/bf16", 2),
    ("8-bit idealizado", 1),
]:
    value = kv_cache_bytes(
        batch_size=1,
        num_layers=32,
        context_length=131072,
        num_kv_heads=32,
        head_dim=128,
        bytes_per_value=bytes_per_value,
    )

    precision_rows.append({
        "precision": precision,
        "bytes_por_valor_idealizado": bytes_per_value,
        "GB_MHA": to_gb(value),
        "GiB_MHA": to_gib(value),
    })

pd.DataFrame(precision_rows)

#### **12. VALIDATE**

Si `distilgpt2` está disponible localmente:

```text
fórmula -> configuración real del modelo -> past_key_values materializado -> comparación
```

Para `distilgpt2` los valores se leen desde `model.config`; no se hardcodean.

La suma de `numel * element_size` mide el **payload tensorial materializado** del cache. No mide toda la memoria física reservada por PyTorch o CUDA.

In [ ]:
ALLOW_DOWNLOAD = False
MODEL_NAME = "distilgpt2"

HF_READY = False
tokenizer = None
model = None

try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
    )

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME,
            local_files_only=not ALLOW_DOWNLOAD,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            local_files_only=not ALLOW_DOWNLOAD,
        ).to(DEVICE)
        model.eval()

        if tokenizer.pad_token is None:
            tokenizer.pad_token = (
                tokenizer.eos_token
            )

        HF_READY = True
        print("Modelo listo:", MODEL_NAME)
    except Exception as exc:
        print(
            "Modelo no disponible localmente."
        )
        print(
            "Cambia ALLOW_DOWNLOAD=True "
            "si deseas descargarlo."
        )
        print(
            "Detalle:",
            type(exc).__name__,
        )
except Exception as exc:
    print(
        "transformers no está instalado."
    )
    print(
        "Detalle:",
        type(exc).__name__,
    )

print("HF_READY =", HF_READY)

In [ ]:
def _nested_tensor_bytes(
    value,
) -> int:
    """Cuenta bytes en tensores contenidos en estructuras K/V anidadas."""
    if value is None:
        return 0

    if torch.is_tensor(value):
        return (
            value.nelement()
            * value.element_size()
        )

    if isinstance(value, dict):
        return sum(
            _nested_tensor_bytes(item)
            for item in value.values()
        )

    if isinstance(value, (tuple, list)):
        return sum(
            _nested_tensor_bytes(item)
            for item in value
        )

    return 0


def cache_payload_bytes(
    cache,
    model_name: str,
) -> int:
    """
    Suma el payload tensorial materializado de K/V.

    Soporta:
    - caches modernos con .layers;
    - capas con .keys/.values tensoriales o anidados;
    - caches con .key_cache/.value_cache;
    - caches legacy tuple/list.
    """
    if cache is None:
        return 0

    # API moderna: Cache con capas.
    if hasattr(cache, "layers"):
        total = 0

        for layer in cache.layers:
            total += _nested_tensor_bytes(
                getattr(layer, "keys", None)
            )
            total += _nested_tensor_bytes(
                getattr(layer, "values", None)
            )

        if total > 0:
            return total

    # Algunas implementaciones exponen listas K/V a nivel del cache.
    if (
        hasattr(cache, "key_cache")
        or hasattr(cache, "value_cache")
    ):
        total = (
            _nested_tensor_bytes(
                getattr(cache, "key_cache", None)
            )
            + _nested_tensor_bytes(
                getattr(cache, "value_cache", None)
            )
        )

        if total > 0:
            return total

    # Compatibilidad con caches legacy.
    if isinstance(cache, (tuple, list, dict)):
        total = _nested_tensor_bytes(cache)

        if total > 0:
            return total

    raise TypeError(
        "No se pudo identificar el payload K/V "
        f"para MODEL_NAME={model_name!r}; "
        f"tipo_cache={type(cache).__name__}. "
        "Inspecciona la estructura del cache antes de "
        "compararla con la fórmula analítica."
    )


def config_value(
    config,
    *names,
):
    for name in names:
        value = getattr(config, name, None)

        if value is not None:
            return value

    raise AttributeError(
        f"No se encontró ninguno de: {names}"
    )


if HF_READY:
    prompt = "Efficient language model inference"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(
            **inputs,
            use_cache=True,
        )

    cache = outputs.past_key_values

    num_layers = int(
        config_value(
            model.config,
            "num_hidden_layers",
            "n_layer",
        )
    )
    num_heads = int(
        config_value(
            model.config,
            "num_attention_heads",
            "n_head",
        )
    )
    num_kv_heads = int(
        getattr(
            model.config,
            "num_key_value_heads",
            num_heads,
        )
    )
    hidden_size = int(
        config_value(
            model.config,
            "hidden_size",
            "n_embd",
        )
    )
    head_dim = hidden_size // num_heads

    dtype_bytes = next(
        model.parameters()
    ).element_size()

    predicted = kv_cache_bytes(
        batch_size=int(
            inputs["input_ids"].shape[0]
        ),
        num_layers=num_layers,
        context_length=int(
            inputs["input_ids"].shape[1]
        ),
        num_kv_heads=num_kv_heads,
        head_dim=head_dim,
        bytes_per_value=dtype_bytes,
    )

    materialized = cache_payload_bytes(
        cache,
        model_name=MODEL_NAME,
    )

    ratio = (
        materialized / predicted
        if predicted > 0
        else float("nan")
    )

    # Este check es fuerte solo cuando el baseline esperado es
    # full MHA con cache estándar y sin compresión/truncamiento.
    standard_full_mha = (
        num_kv_heads == num_heads
        and "distilgpt2" in MODEL_NAME.lower()
    )

    ratio_close = (
        math.isfinite(ratio)
        and abs(ratio - 1.0) < 0.05
    )

    comparison = pd.DataFrame([{
        "modelo": MODEL_NAME,
        "tipo_cache": type(cache).__name__,
        "layers": num_layers,
        "attention_heads": num_heads,
        "kv_heads": num_kv_heads,
        "head_dim": head_dim,
        "tokens_prompt": int(
            inputs["input_ids"].shape[1]
        ),
        "bytes_por_valor": dtype_bytes,
        "bytes_formula": predicted,
        "bytes_cache_materializado": materialized,
        "ratio_materializado_formula": ratio,
        "baseline_comparable": standard_full_mha,
        "ratio_cerca_de_1": ratio_close,
    }])

    display(comparison)

    if standard_full_mha:
        assert ratio_close, (
            "La fórmula y el payload materializado deberían "
            "ser cercanos para este baseline. "
            f"MODEL_NAME={MODEL_NAME!r}, "
            f"tipo_cache={type(cache).__name__}, "
            f"ratio={ratio:.4f}. "
            "Revisa longitud efectiva del cache, dtype, "
            "layout K/V y configuración del modelo."
        )
    else:
        print(
            "Nota: no se aplica un assert ratio≈1 porque "
            "el tipo de cache o la arquitectura puede "
            "usar truncamiento, cuantización, capacidad "
            "preasignada u otro layout."
        )
else:
    print(
        "Validación con cache real omitida: "
        "HF_READY=False."
    )

#### **13. OPCIONAL**

Si `HF_READY=True`, compara:

```text
prefill_forward vs decode_forward_por_token
```

Para `prefill` se realizan **7 repeticiones** después de un warm-up.

Para ambas mediciones se reporta:

```text
n
mediana
Q1
Q3
IQR
mínimo
máximo
```

El objetivo del IQR es hacer visible la variabilidad de timing, especialmente en CPU compartida.

No se fija una nueva seed para el timing porque no se está evaluando una trayectoria estocástica: el prompt y la operación medida permanecen fijos.

Esto es una medición local de `forward`.

No se denomina automáticamente TTFT o ITL end-to-end porque no incluye necesariamente tokenización, scheduling, serving, transferencias y otros componentes del sistema.

En GPU se sincroniza antes y después de medir para evitar confundir lanzamiento asíncrono con tiempo ejecutado.

In [ ]:
def synchronize_if_needed() -> None:
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def summarize_ms(
    values: list[float],
) -> dict:
    if len(values) < 4:
        raise ValueError(
            "Se requieren al menos 4 mediciones "
            "para reportar cuartiles."
        )

    q1, _, q3 = statistics.quantiles(
        values,
        n=4,
        method="inclusive",
    )

    return {
        "n": len(values),
        "mediana_ms": float(
            statistics.median(values)
        ),
        "q1_ms": float(q1),
        "q3_ms": float(q3),
        "iqr_ms": float(q3 - q1),
        "min_ms": float(min(values)),
        "max_ms": float(max(values)),
    }


def measure_prefill_forward_ms(
    prompt: str,
    repeats: int = 7,
) -> dict:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(DEVICE)

    # Warm-up fuera de la medición.
    with torch.no_grad():
        _ = model(
            **inputs,
            use_cache=True,
        )

    times = []

    for _ in range(repeats):
        synchronize_if_needed()
        t0 = time.perf_counter()

        with torch.no_grad():
            _ = model(
                **inputs,
                use_cache=True,
            )

        synchronize_if_needed()

        times.append(
            1000.0
            * (
                time.perf_counter()
                - t0
            )
        )

    return summarize_ms(times)


def measure_decode_forward_ms(
    prompt: str,
    new_tokens: int = 8,
) -> dict:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(
            **inputs,
            use_cache=True,
        )

    past = outputs.past_key_values

    next_token = outputs.logits[
        :,
        -1,
        :,
    ].argmax(
        dim=-1,
        keepdim=True,
    )

    attention_mask = inputs.get(
        "attention_mask",
        torch.ones_like(
            inputs["input_ids"]
        ),
    )

    step_times = []

    for _ in range(new_tokens):
        attention_mask = torch.cat(
            [
                attention_mask,
                torch.ones(
                    (
                        attention_mask.shape[0],
                        1,
                    ),
                    dtype=attention_mask.dtype,
                    device=attention_mask.device,
                ),
            ],
            dim=1,
        )

        synchronize_if_needed()
        t0 = time.perf_counter()

        with torch.no_grad():
            outputs = model(
                input_ids=next_token,
                attention_mask=attention_mask,
                past_key_values=past,
                use_cache=True,
            )

        synchronize_if_needed()

        step_times.append(
            1000.0
            * (
                time.perf_counter()
                - t0
            )
        )

        past = outputs.past_key_values

        next_token = outputs.logits[
            :,
            -1,
            :,
        ].argmax(
            dim=-1,
            keepdim=True,
        )

    return summarize_ms(step_times)


if HF_READY:
    prompt = "Efficient language model inference"

    prefill_stats = measure_prefill_forward_ms(
        prompt,
        repeats=7,
    )

    decode_stats = measure_decode_forward_ms(
        prompt,
        new_tokens=8,
    )

    timing_table = pd.DataFrame([
        {
            "fase": "prefill_forward",
            **prefill_stats,
            "dispositivo": DEVICE,
        },
        {
            "fase": "decode_forward_por_token",
            **decode_stats,
            "dispositivo": DEVICE,
        },
    ])

    display(timing_table)
else:
    print(
        "Timing prefill/decode omitido: "
        "HF_READY=False."
    )

#### **14. CRITIQUE**

Clasifica como **defendible**, **no defendible** o **depende**:

1. "GQA usa 1/4 del componente K/V modelado, por tanto usa 1/4 de toda la memoria GPU."
2. "FlashAttention reduce necesariamente el tamaño lógico del KV cache."
3. "Menor `repeticion_bigramas` demuestra mayor calidad."
4. "La entropía inicial debe variar entre seeds."
5. "La entropía media de trayectoria puede variar entre seeds."
6. "El payload de `past_key_values` equivale a toda la memoria física de GPU."
7. "Menor `prefill_forward_ms` garantiza menor TTFT de un servidor."
8. "Una estimación 8-bit divide exactamente entre dos toda la memoria física respecto de fp16."
9. "Un ratio fórmula/cache distinto de 1 siempre significa que la fórmula es incorrecta."
10. "La tasa de aceptación en speculative decoding depende de qué tan bien aproxima `q` a `p`."

El laboratorio experimental queda cerrado antes de las exposiciones.

#### **17. Artefacto de apoyo - Speculative Decoding**

Este microartefacto **no implementa un servidor ni drafting de varios tokens**.

Ilustra la regla probabilística de aceptación de **un token propuesto por un draft model**:

```text
x ~ q
```

donde `q` es la distribución draft y `p` la distribución del modelo objetivo.

El token se acepta con probabilidad:

$$
\alpha(x)
=
\min\left(
1,
\frac{p(x)}{q(x)}
\right).
$$

Si se rechaza, se muestrea de la distribución residual proporcional a:

$$
\max(p-q,0).
$$

Esto es deliberadamente más fiel al mecanismo que un simulador Bernoulli con probabilidad fija de aceptación.

La actividad permite discutir:

- por qué la tasa de aceptación depende de la cercanía entre `p` y `q`,
- por qué mayor aceptación puede reducir trabajo del modelo objetivo,
- por qué esto no modifica por sí mismo el KV cache ni demuestra menor latencia end-to-end.

In [ ]:
def speculative_single_token(
    target_probs: torch.Tensor,
    draft_probs: torch.Tensor,
) -> dict:
    """Un paso mínimo de speculative sampling."""
    p = target_probs/target_probs.sum()
    q = draft_probs/draft_probs.sum()

    draft_token = int(
        torch.multinomial(
            q,
            num_samples=1,
        ).item()
    )

    acceptance_prob = min(
        1.0,
        float(
            p[draft_token]
            / q[draft_token]
        ),
    )

    accepted = (
        random.random()
        < acceptance_prob
    )

    if accepted:
        output_token = draft_token
        source = "draft_accepted"
    else:
        residual = torch.clamp(
            p - q,
            min=0.0,
        )

        # Si p==q numéricamente, el residual puede ser cero.
        if float(residual.sum()) <= 1e-12:
            residual = p.clone()
        else:
            residual = (
                residual
                / residual.sum()
            )

        output_token = int(
            torch.multinomial(
                residual,
                num_samples=1,
            ).item()
        )
        source = "residual_sample"

    return {
        "draft_token": draft_token,
        "acceptance_prob": acceptance_prob,
        "accepted": accepted,
        "output_token": output_token,
        "source": source,
    }


target = torch.tensor(
    [0.55, 0.25, 0.15, 0.05],
    dtype=torch.float32,
)

draft_close = torch.tensor(
    [0.50, 0.30, 0.15, 0.05],
    dtype=torch.float32,
)

draft_far = torch.tensor(
    [0.20, 0.20, 0.20, 0.40],
    dtype=torch.float32,
)


def estimate_acceptance_rate(
    target_probs: torch.Tensor,
    draft_probs: torch.Tensor,
    trials: int = 2000,
) -> float:
    accepted = 0

    for _ in range(trials):
        result = speculative_single_token(
            target_probs,
            draft_probs,
        )
        accepted += int(
            result["accepted"]
        )

    return accepted / trials


random.seed(SEED)
torch.manual_seed(SEED)

spec_table = pd.DataFrame([
    {
        "draft": "cercano",
        "acceptance_rate": (
            estimate_acceptance_rate(
                target,
                draft_close,
            )
        ),
    },
    {
        "draft": "lejano",
        "acceptance_rate": (
            estimate_acceptance_rate(
                target,
                draft_far,
            )
        ),
    },
])

spec_table

#### **18. EXPOSE/DEFEND**

Las exposiciones se realizan **al final**.

Formato por grupo:

```text
7 min de exposición

3 min de defensa
```

Por tanto:

```text
10 min por grupo
```

El bloque disponible es de hasta 100 minutos.

```text
máximo teórico dentro de esta sesión = 10 grupos
```

Si existen menos grupos, la clase termina antes.

No se extiende el laboratorio para consumir tiempo sobrante.

#### **Temas base**

1. FlashAttention.
2. MQA/GQA.
3. MLA.
4. SWA y contexto largo.

Temas avanzados si existen más grupos:

5. PagedAttention.
6. Speculative Decoding.

Si hay más grupos que temas, dos grupos pueden trabajar el mismo método con claims, papers relacionados o limitaciones diferentes.

#### **Estructura obligatoria**

Cada exposición contiene:

```text
problema -> baseline -> mecanismo -> recurso afectado -> evidencia -> limitación -> conexión con el laboratorio
```

Además debe mostrar:

- herramienta 1,
- herramienta 2,
- seed paper,
- fuente primaria,
- claim técnico,
- una fuente relacionada o citante,
- una limitación.
